In [0]:
import pyspark.sql.functions as f
from delta.tables import DeltaTable


#Paramêtros construção de tabelas
dbutils.widgets.text("tabela_origem", "zeta_project.bronze.tabela", "1. Tabela Bronze")
dbutils.widgets.text("tabela_destino", "zeta_project.silver.tabela", "2. Tabela Silver")


# Parametros para tratamento de colunas()
dbutils.widgets.text("col_inteiro", "", "Colunas Inteiras (separadas por vírgula)")
dbutils.widgets.text("col_decimal", "", "Colunas Decimais (separadas por vírgula)")
dbutils.widgets.text("col_timestamp", "", "Colunas Decimais (separadas por vírgula)")
dbutils.widgets.text("chave", "", "Coleta as chaves para tratamento")
dbutils.widgets.text("col_texto", "", "Colunas Texto (separadas por vírgula)")

tabela_origem = dbutils.widgets.get("tabela_origem")
tabela_destino = dbutils.widgets.get("tabela_destino")




In [0]:
#---------------------------------------------------------------------------------------#
# BLOCO DE TRATAMENTO                                                                   #
#---------------------------------------------------------------------------------------#



#Leitura da tabela bronze
df = spark.read.table(tabela_origem)

# Cria lista com os tipos das colunas passada através de parametros

cols_int = [ci.strip() for ci in dbutils.widgets.get("col_inteiro").split(",") if ci.strip()]
cols_dec = [cd.strip() for cd in dbutils.widgets.get("col_decimal").split(",") if cd.strip()]
cols_times = [ct.strip() for ct in dbutils.widgets.get("col_timestamp").split(",") if ct.strip()]
cols_texto = [ct.strip() for ct in dbutils.widgets.get("col_texto").split(",") if ct.strip()]


# Captura a pk simples ou composta para tratamento
chaves = [ck.strip() for ck in dbutils.widgets.get("chave").split(",") if ck.strip()]



# Tenta realizar as conversões solicitadas
mapa_numeros = {"one": "1", "two": "2", "three": "3", "four": "4"}

for ci in cols_int:
    if ci in df.columns:
        df = df.withColumn(ci, f.lower(f.trim(f.col(ci))))
        
        df = df.replace(mapa_numeros, subset=[ci])
        
        df = df.withColumn(ci, f.col(ci).cast("int"))
        
for cd in cols_dec:
    if cd in df.columns:     
 
        df = df.withColumn(cd, 
            f.expr(f"""
                CASE 
                    WHEN {cd} LIKE '%,%' THEN try_cast(replace(replace({cd}, '.', ''), ',', '.') AS decimal(18,2))
                    ELSE try_cast({cd} AS decimal(18,2))
                END
            """)
        )



for ct in cols_times:
    if ct in df.columns:
        df = df.withColumn(ct, f.col(ct))        
        ct_limpa = f.trim(f.col(ct))
# Inclusao de tentativa de cast devido a diferentes tipos de datas encontradas
        df = df.withColumn(ct, 
            f.coalesce(
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-M-d H:m:s")), 
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-MM-dd HH:mm:ss")),
                f.try_to_timestamp(ct_limpa, f.lit("yyyy-MM-dd HH:mm")),
                f.try_to_timestamp(ct_limpa, f.lit("d/M/yyyy H:m:s")),
                f.try_to_timestamp(ct_limpa, f.lit("dd/MM/yyyy HH:mm")),
                f.try_to_timestamp(ct_limpa, f.lit("d/M/yyyy H:m")),
                f.try_to_timestamp(ct_limpa)
            )
        )


df = df.dropna(how="any", subset=chaves)

df = df.fillna(value=0, subset=cols_int)

df = df.fillna(value=0.0, subset=cols_dec)

df = df.fillna(value="Não informado", subset=cols_texto)


In [0]:
#--------------------------------------------------------#
# Dropa duplicadas e gera tabela com merge upsert         #
#--------------------------------------------------------#

df = df.dropDuplicates(subset=chaves)

df.createOrReplaceTempView("vw_merge")

if not spark.catalog.tableExists(tabela_destino):
    df.write.format("delta").mode("overwrite").clusterBy(*chaves).saveAsTable(tabela_destino)
else:
    condicao_join = " AND ".join([f"t.{c} = s.{c}" for c in chaves])

    query_merge = f"""
        MERGE INTO {tabela_destino} as t 
        USING vw_merge as s 
        ON {condicao_join}
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """

    spark.sql(query_merge)

